# 03 — Onset CNN (EXP-020), Colab / PyTorch

A fully-convolutional CNN predicts a per-frame **onset activation** from log-mel
frames; our **own** LFSF peak picker (`OnsetDetector._pick`) turns it into onset
times. No `librosa.onset.onset_detect`, no library peak-picker, no madmom — the
musical decision (peak picking) stays our code.

**Fair test first** (`TRAIN_ON_EXTRA_ONLY=True`): train on the 150
`train_extra_onsets` files, evaluate on the 127 main (c127, test-like) the model
never saw. The bar is fusion's c127 (0.8055, leakage-inflated) and the leaderboard
onset 0.775. If it clears that, retrain on all 277 (set the toggle False) for the
shippable model and we wire `OnsetDetector` after review.

Run in Colab (GPU). Upload `train.zip` + `train_extra_onsets.zip` to
`MyDrive/amp_data/`.


In [ ]:
# === Setup (Colab-ready) ===
import sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install",
                    "librosa", "soundfile", "mir_eval"], check=False)
    REPO = Path("amp-challenge")
    if not REPO.exists():
        subprocess.run(["git", "clone",
                        "https://github.com/8asic/amp2026-onset-beat-tempo.git",
                        str(REPO)], check=True)
else:
    REPO = Path.cwd().parent
sys.path.insert(0, str(REPO))

import numpy as np
import librosa
import mir_eval
import torch
import torch.nn as nn

from src.config import config
from src.detectors import OnsetDetector
from src.utils import load_onsets_gt

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", DEVICE, "| repo:", REPO)

In [ ]:
# === Data: extract zips from Drive to fast local disk, then locate dirs ===
import zipfile

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ZIP_DIR = Path("/content/drive/MyDrive/amp_data")   # <-- folder with the zips
    WORK = Path("/content/data"); WORK.mkdir(exist_ok=True)
    for name in ["train", "train_extra_onsets"]:
        z, dest = ZIP_DIR / f"{name}.zip", WORK / name
        if dest.exists():
            print("already extracted:", dest); continue
        assert z.exists(), f"Missing {z} - upload {name}.zip to {ZIP_DIR}"
        print("extracting", z, "...")
        with zipfile.ZipFile(z) as zf:
            zf.extractall(dest)
    base = WORK
else:
    base = REPO / "data" / "processed"

def find_dir_with(suffix, root):
    root = Path(root)
    if not root.exists():
        return None
    for p in [root] + [d for d in root.rglob("*") if d.is_dir()]:
        if any(p.glob(f"*{suffix}")):
            return p
    return None

train_dir = find_dir_with(".onsets.gt", base / "train")
extra_dir = find_dir_with(".onsets.gt", base / "train_extra_onsets")
print("train_dir:", train_dir)
print("extra_dir:", extra_dir)
assert train_dir and extra_dir, "Could not locate .onsets.gt dirs - check ZIP_DIR / uploads"

In [ ]:
# === Collect onset-annotated files (127 main + 150 extra = 277) ===
def collect(d, corpus):
    d = Path(d); items = []
    if not d.exists():
        return items
    for wav in sorted(d.glob("*.wav")):
        gtp = d / f"{wav.stem}.onsets.gt"
        if gtp.exists():
            ons = load_onsets_gt(gtp)
            if ons is not None and len(ons):
                items.append((str(wav), np.asarray(ons, dtype=float), corpus))
    return items

main_files = collect(train_dir, "c127")          # 127 main (test-like)
extra_files = collect(extra_dir, "extra")        # 150 supplementary
files = main_files + extra_files
print(f"main(c127): {len(main_files)}  extra: {len(extra_files)}  total: {len(files)}")
assert files, "No files found - check DATA_ROOT path."

In [ ]:
# === Features: MULTI-RESOLUTION log-mel (3 STFT windows) + onset labels (cached) ===
# Schluter & Boeck SOTA onset recipe: stack 3 mel-spectrograms at different STFT
# window sizes as input channels -> the net sees fine TIME (small window) and fine
# FREQUENCY (large window) resolution at once. Same hop -> aligned frames.
SR = config.audio.sample_rate          # 22050
HOP = config.audio.onset_hop_length    # 256
NFFTS = [1024, 2048, 4096]             # multi-resolution windows
NMELS = config.audio.onset_n_mels      # 82
FMIN, FMAX = config.audio.onset_fmin, config.audio.onset_fmax
FPS = SR / HOP
LABEL_W = 1

def multilogmel(y):
    specs = []
    for nfft in NFFTS:
        m = librosa.feature.melspectrogram(y=y, sr=SR, n_fft=nfft, hop_length=HOP,
                                           n_mels=NMELS, fmin=FMIN, fmax=FMAX)
        specs.append(np.log1p(m).T)            # (T, NMELS)
    T = min(s.shape[0] for s in specs)
    return np.stack([s[:T] for s in specs], axis=0).astype(np.float32)  # (3, T, NMELS)

def onset_labels(onsets, T):
    lab = np.zeros(T, dtype=np.float32)
    for f in np.round(onsets * FPS).astype(int):
        for dd in range(-LABEL_W, LABEL_W + 1):
            if 0 <= f + dd < T:
                lab[f + dd] = 1.0
    return lab

CACHE = REPO / "experiments" / ".cache_onset_mr"
CACHE.mkdir(parents=True, exist_ok=True)

data = []
for i, (wav, onsets, corpus) in enumerate(files):
    stem = Path(wav).stem
    cp = CACHE / f"{stem}.npz"
    if cp.exists():
        d = np.load(cp); X, y = d["X"], d["y"]
    else:
        y_audio, _ = librosa.load(wav, sr=SR)
        X = multilogmel(y_audio)
        y = onset_labels(onsets, X.shape[1])
        np.savez(cp, X=X, y=y)
    data.append({"stem": stem, "X": X, "y": y, "onsets": onsets, "wav": wav, "corpus": corpus})
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{len(files)} features")
print("features ready:", len(data), "| multi-res", NFFTS, "| X shape", data[0]["X"].shape)

In [ ]:
# === Split + standardization (per channel+mel, stats from TRAIN only) ===
# Modes (TRAIN_ALL overrides): ship on ALL 277 (incl every c127 file), fixed epochs.
TRAIN_ALL = True              # BEST-OF-BEST ship model: train on every file
TRAIN_ON_EXTRA_ONLY = False   # fair test (train extra-only, eval c127)
if TRAIN_ALL:
    train_set = list(data); val_set = []
elif TRAIN_ON_EXTRA_ONLY:
    train_set = [d for d in data if d["corpus"] == "extra"]
    val_set   = [d for d in data if d["corpus"] == "c127"]
else:
    rng = np.random.default_rng(0)
    order = rng.permutation(len(data)); n_val = int(0.2 * len(data))
    vi = set(order[:n_val].tolist())
    train_set = [data[i] for i in range(len(data)) if i not in vi]
    val_set = [data[i] for i in range(len(data)) if i in vi]
print(f"train {len(train_set)}  val {len(val_set)}  (TRAIN_ALL={TRAIN_ALL}, extra_only={TRAIN_ON_EXTRA_ONLY})")
assert train_set, "empty train"

allX = np.concatenate([d["X"] for d in train_set], axis=1)   # (3, sumT, NMELS)
MU = allX.mean(axis=1, keepdims=True)                        # (3, 1, NMELS)
SD = allX.std(axis=1, keepdims=True) + 1e-8
def norm(X):
    return (X - MU) / SD

In [ ]:
# === EXP-022 augmentation (TRAIN only): pitch-shift + time-stretch (multi-res) ===
VARIANTS = [(1.0, 0), (0.90, 0), (1.11, 0), (1.0, -2), (1.0, 2), (0.94, 1), (1.06, -1)]
AUG = REPO / "experiments" / ".cache_onset_mr_aug"
AUG.mkdir(parents=True, exist_ok=True)

def aug_feat(wav, onsets, rate, pitch):
    cp = AUG / f"{Path(wav).stem}_r{int(round(rate*100))}_p{pitch}.npz"
    if cp.exists():
        d = np.load(cp); return d["X"], d["y"]
    y, _ = librosa.load(wav, sr=SR)
    if abs(rate - 1.0) > 1e-6:
        y = librosa.effects.time_stretch(y, rate=rate)
    if pitch != 0:
        y = librosa.effects.pitch_shift(y, sr=SR, n_steps=pitch)
    X = multilogmel(y)
    yb = onset_labels(onsets / rate, X.shape[1])
    X = X.astype(np.float32); yb = yb.astype(np.float32)
    np.savez(cp, X=X, y=yb)
    return X, yb

train_items = []
for i, d in enumerate(train_set):
    for (r, pch) in VARIANTS:
        X, yb = aug_feat(d["wav"], d["onsets"], r, pch)
        train_items.append({"X": X, "y": yb})
    if (i + 1) % 50 == 0:
        print(f"  aug {i+1}/{len(train_set)}  ({len(train_items)} seqs)")
val_items = val_set
print(f"train {len(train_set)} -> {len(train_items)} aug seqs | val {len(val_items)} clean")

In [ ]:
# === Dataset / loaders (train = pitch/time-augmented, val = clean) ===
# EXP-022: pitch+time augmentation (the RIGHT kind) for the data-limited onset
# CNN. Small model kept (EXP-020 showed bigger overfits on 277 raw files; the fix
# is more DATA via augmentation, not capacity).
class OnsetDS(torch.utils.data.Dataset):
    def __init__(self, items): self.items = items
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        d = self.items[i]
        return (torch.from_numpy(norm(d["X"])), torch.from_numpy(d["y"]))

train_dl = torch.utils.data.DataLoader(OnsetDS(train_items), batch_size=1, shuffle=True)
val_dl = torch.utils.data.DataLoader(OnsetDS(val_items), batch_size=1, shuffle=False)

In [ ]:
# === Model: multi-resolution onset CNN (3-channel input) ===
class OnsetCNN(nn.Module):
    def __init__(self, n_mels, in_ch=3):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, 16, (3, 3), padding=(1, 1)), nn.BatchNorm2d(16), nn.ReLU(),
            nn.MaxPool2d((1, 3)),
            nn.Conv2d(16, 32, (3, 3), padding=(1, 1)), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d((1, 3)),
            nn.Conv2d(32, 32, (3, 3), padding=(1, 1)), nn.BatchNorm2d(32), nn.ReLU(),
        )
        with torch.no_grad():
            f = self.conv(torch.zeros(1, in_ch, 8, n_mels)).shape[-1]
        self.head = nn.Sequential(
            nn.Linear(32 * f, 64), nn.ReLU(), nn.Dropout(0.3), nn.Linear(64, 1))

    def forward(self, x):                 # x: (B, 3, T, n_mels)
        h = self.conv(x)                  # (B, 32, T, f)
        B, C, T, F = h.shape
        h = h.permute(0, 2, 1, 3).reshape(B, T, C * F)
        return self.head(h).squeeze(-1)

model = OnsetCNN(NMELS).to(DEVICE)
print(sum(p.numel() for p in model.parameters()), "params")

In [ ]:
# === Train (BCE pos-weighted + grad-clip + scheduler; early-stop if val) ===
POS_WEIGHT = torch.tensor([4.0], device=DEVICE)
crit = nn.BCEWithLogitsLoss(pos_weight=POS_WEIGHT)
opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=3)
HAS_VAL = len(val_items) > 0
EPOCHS, ES_PATIENCE = (60, 8) if HAS_VAL else (16, 10**9)   # train-all: fixed 16 epochs

best_val, best_state, since_best = float("inf"), None, 0
for ep in range(1, EPOCHS + 1):
    model.train(); tr = 0.0
    for X, y in train_dl:
        X, y = X.to(DEVICE), y.to(DEVICE)
        opt.zero_grad()
        loss = crit(model(X), y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=3.0)
        opt.step()
        tr += loss.item()
    tr /= len(train_dl)
    if HAS_VAL:
        model.eval(); va = 0.0
        with torch.no_grad():
            for X, y in val_dl:
                X, y = X.to(DEVICE), y.to(DEVICE)
                va += crit(model(X), y).item()
        va /= len(val_dl)
        sched.step(va)
        if va < best_val:
            best_val, since_best = va, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            since_best += 1
        print(f"epoch {ep:2d}  train {tr:.4f}  val {va:.4f}  lr {opt.param_groups[0]['lr']:.2e}")
        if since_best >= ES_PATIENCE:
            print(f"early stop at epoch {ep}"); break
    else:
        sched.step(tr)   # no held-out: schedule on train loss
        print(f"epoch {ep:2d}  train {tr:.4f}  lr {opt.param_groups[0]['lr']:.2e}  (train-all)")

if HAS_VAL and best_state is not None:
    model.load_state_dict(best_state)
print(f"best val loss {best_val:.4f}" if HAS_VAL else f"trained on all {len(train_items)} seqs, {EPOCHS} epochs")

In [ ]:
# === Decode with OUR peak picker; eval by corpus vs fusion baseline ===
od = OnsetDetector()

@torch.no_grad()
def activation(X):
    model.eval()
    t = torch.from_numpy(norm(X)).unsqueeze(0).to(DEVICE)
    return torch.sigmoid(model(t)).squeeze(0).cpu().numpy()

if not val_set:
    print("TRAIN_ALL: no held-out eval (ship model trained on all 277). Evaluate via leaderboard.")
else:
    def f1_at(acts, delta):
        fs = {"c127": [], "extra": []}
        for d, a in zip(val_set, acts):
            peaks = od._pick(np.asarray(a, dtype=np.float64), FPS, delta)
            est = np.array(peaks, dtype=int) / FPS
            f = mir_eval.onset.f_measure(d["onsets"], est, window=0.05)[0] if len(est) else 0.0
            fs[d["corpus"]].append(f)
        return fs
    acts = [activation(d["X"]) for d in val_set]
    print("CNN onset F1 by peak-pick delta (val):")
    best = (None, -1.0)
    for delta in [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]:
        fs = f1_at(acts, delta)
        allf = fs["c127"] + fs["extra"]
        m = float(np.mean(allf))
        c127 = float(np.mean(fs["c127"])) if fs["c127"] else 0.0
        print(f"  delta={delta:.2f}: all={m:.4f}  c127={c127:.4f} ({len(fs['c127'])})  "
              f"extra={np.mean(fs['extra']) if fs['extra'] else 0:.4f}")
        if c127 > best[1]:
            best = (delta, c127)
    print("")
    print(f"BEST c127: delta={best[0]}  F1={best[1]:.4f}")

In [ ]:
# === Save weights + everything inference needs ===
fname = "onset_cnn_extra.pt" if TRAIN_ON_EXTRA_ONLY else "onset_cnn.pt"
out = REPO / "models" / fname
out.parent.mkdir(parents=True, exist_ok=True)
torch.save({
    "state_dict": model.state_dict(),
    "mu": MU, "sd": SD,
    "n_mels": NMELS, "sr": SR, "hop": HOP, "n_ffts": NFFTS,
    "fmin": FMIN, "fmax": FMAX, "label_w": LABEL_W, "multi_res": True,
}, out)
print("saved", out, "| multi_res NFFTS", NFFTS)
# In Colab: from google.colab import files; files.download(str(out))